# 01 — Eulerian Video Magnification: Step-by-Step Walkthrough

**Eulerian Video Magnification (EVM)** amplifies small intensity changes at each pixel that would otherwise be invisible to the naked eye.  The core idea is:

1. Decompose every frame into a **Laplacian spatial pyramid** — different levels capture different spatial scales (fine detail at the top, coarse structure at the bottom).
2. For each pyramid level, collect the time-series of pixel intensities across all frames.
3. **Bandpass-filter** that time-series to isolate a specific frequency range (e.g., 0.5–2 Hz for a pulse signal).
4. Multiply the filtered signal by an amplification factor **α** and add it back to the original.
5. Collapse the modified pyramid to reconstruct each frame.

> **Run this notebook from the project root directory** so that `from src.X import Y` imports resolve correctly.
>
> Set `VIDEO_PATH` in the *Configuration* cell below before running.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

# Ensure project root is on sys.path whether launched from project root or notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from src.io.video_io import load_video, save_video
from src.utils.phase_utils import bgr2yiq, yiq2rgb
from src.pyramids.spatial import build_laplacian_pyramid, collapse_laplacian_pyramid
from src.filters.temporal import bandpass_filter_butter
from src.magnification.eulerian import run_eulerian

print('Imports OK')

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
# Set VIDEO_PATH to video file before running.
VIDEO_PATH   = str(PROJECT_ROOT / 'videos' / 'wrist.avi')

FREQ_LOW     = 0.5    # Hz  — lower temporal bandpass cutoff
FREQ_HIGH    = 2.0    # Hz  — upper temporal bandpass cutoff
ALPHA        = 50.0   # amplification factor
LEVELS       = 4      # Laplacian pyramid levels
SCALE        = 1.0    # spatial scale factor (< 1 downsamples)
OUTPUT_PATH  = str(PROJECT_ROOT / 'results' / 'videos' / 'evm_output.mp4')
# ──────────────────────────────────────────────────────────────────────────

In [ ]:
# Load video
frames, fps = load_video(VIDEO_PATH, scale=SCALE)
T, H, W, C = frames.shape
print(f'{T} frames  |  {W}×{H} px  |  {fps:.2f} fps')

# Show first frame
fig, ax = plt.subplots(figsize=(6, 4))
ax.imshow(cv2.cvtColor((frames[0] * 255).astype('uint8'), cv2.COLOR_BGR2RGB))
ax.set_title('Frame 0 (original)')
ax.axis('off')
plt.tight_layout()
plt.show()

## Step 1 — Convert to YIQ Colorspace

We process only the **luma (Y) channel** of the YIQ colorspace.  Luma captures brightness variation (where the physiological signal lives) while the I and Q chroma channels are left unchanged to prevent colour fringing in the output.

In [ ]:
yiq0 = bgr2yiq(frames[0])   # (H, W, 3) — Y, I, Q

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
labels = ['Y (luma)', 'I (orange-blue)', 'Q (purple-green)']
cmaps  = ['gray', 'RdBu_r', 'PRGn']
for i, (ax, label, cmap) in enumerate(zip(axes, labels, cmaps)):
    ax.imshow(yiq0[:, :, i], cmap=cmap)
    ax.set_title(label)
    ax.axis('off')
plt.suptitle('YIQ channels — frame 0', y=1.01)
plt.tight_layout()
plt.show()

## Step 2 — Laplacian Pyramid Spatial Decomposition

The Laplacian pyramid splits the image into **band-pass spatial frequency layers**.  Each level captures detail at a different scale — level 0 is the finest (high-frequency edges), the last level is a low-pass residual.  During amplification we apply the same **α** to every level.

In [ ]:
luma0 = bgr2yiq(frames[0])[:, :, 0].astype('float32')  # single luma channel
pyramid = build_laplacian_pyramid(luma0, levels=LEVELS)

n_levels = len(pyramid)
fig, axes = plt.subplots(1, n_levels, figsize=(4 * n_levels, 3))
for lvl, (ax, band) in enumerate(zip(axes, pyramid)):
    # Normalise for display
    disp = band - band.min()
    if disp.max() > 0:
        disp /= disp.max()
    ax.imshow(disp, cmap='gray')
    label = f'Level {lvl}' if lvl < LEVELS else 'Residual (low-pass)'
    ax.set_title(f'{label}\n{band.shape[1]}×{band.shape[0]}')
    ax.axis('off')
plt.suptitle(f'Laplacian pyramid ({LEVELS} levels + residual)', y=1.01)
plt.tight_layout()
plt.show()

## Step 3 — Temporal Bandpass Filter on a Pixel's Time Series

For a single pixel location, extract its luma value across all frames and apply the Butterworth bandpass filter.  The filtered signal contains only the frequencies of interest (e.g., the pulse band 0.5–2 Hz).  This is what gets amplified.

In [ ]:
# Choose a pixel near the centre of the frame
py, px = H // 2, W // 2

# Extract luma time-series
luma_all = np.stack([bgr2yiq(frames[t])[:, :, 0] for t in range(T)], axis=0)  # (T, H, W)
raw_signal   = luma_all[:, py, px].astype('float64')
filt_signal  = bandpass_filter_butter(raw_signal.astype('float32'), FREQ_LOW, FREQ_HIGH, fps)

t_axis = np.arange(T) / fps

fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(t_axis, raw_signal, linewidth=0.8, color='steelblue')
axes[0].set_ylabel('Luma value')
axes[0].set_title(f'Raw luma signal at pixel ({px}, {py})')

axes[1].plot(t_axis, filt_signal, linewidth=0.8, color='tomato')
axes[1].set_ylabel('Filtered luma')
axes[1].set_xlabel('Time (s)')
axes[1].set_title(f'Bandpass-filtered [{FREQ_LOW}–{FREQ_HIGH} Hz]')

plt.tight_layout()
plt.show()

print(f'Signal amplitude (raw):      {raw_signal.std():.5f}')
print(f'Signal amplitude (filtered): {filt_signal.std():.5f}')

## Step 4 — Full Eulerian Pipeline

`run_eulerian` chains all of the above steps together and returns the amplified video as a NumPy array.

In [ ]:
amplified = run_eulerian(
    frames=frames,
    fps=fps,
    freq_low=FREQ_LOW,
    freq_high=FREQ_HIGH,
    alpha=ALPHA,
    levels=LEVELS,
)

print(f'Output shape: {amplified.shape}  dtype: {amplified.dtype}')
print(f'Value range:  [{amplified.min():.3f}, {amplified.max():.3f}]')

# Show frame 0 original vs amplified side-by-side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, vid, title in zip(axes, [frames, amplified], ['Original', f'EVM  α={ALPHA}']):
    ax.imshow(cv2.cvtColor((vid[0] * 255).astype('uint8'), cv2.COLOR_BGR2RGB))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Save the amplified video
save_video(amplified, OUTPUT_PATH, fps=fps)
print(f'Saved → {OUTPUT_PATH}')